In [13]:
import os
import random
import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [14]:
DATA_PATH = None
OUT_DIR = "outputs"
RUN_TRANSFORMER = False
SEED = 42

random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

STOPWORDS = ENGLISH_STOP_WORDS - {"not", "no", "never", "nor", "cannot"}

THEMES = {
    "Mentorship & Guidance": ["mentor", "guidance", "supervisor", "manager", "support", "ignored", "unavailable"],
    "Workload & Work-Life Balance": ["workload", "overtime", "hours", "stress", "overwhelming", "deadline", "burnout", "late"],
    "Compensation": ["pay", "stipend", "salary", "unpaid", "compensation", "paid"],
    "Onboarding & Training": ["onboarding", "orientation", "training", "introduction", "documentation", "confusing"],
    "Communication": ["communication", "feedback", "updates", "unclear", "meeting", "informed"],
    "Learning & Meaningful Work": ["learn", "learning", "boring", "repetitive", "menial", "coffee", "skills", "tasks", "project"],
    "Culture & Inclusion": ["culture", "team", "toxic", "included", "colleagues", "environment"],
}

In [15]:
def make_sample_data(n_per_class=300):
    pos = [
        "My mentor was incredibly supportive and gave me clear guidance every week.",
        "The onboarding was well organized and the documentation was helpful.",
        "I learned a lot and worked on a meaningful project that shipped to users.",
        "The team culture was welcoming and I felt included from day one.",
        "Great communication, regular feedback, and I always felt informed.",
        "The workload was balanced and my supervisor respected my time.",
        "The stipend was fair and the learning opportunities were excellent.",
        "I gained real skills and my manager trusted me with important tasks.",
    ]
    neg = [
        "My mentor was unavailable and I rarely received any guidance.",
        "The onboarding was confusing and there was no proper documentation.",
        "The tasks were boring and repetitive, I did not learn much.",
        "The team culture felt toxic and I did not feel included.",
        "Communication was poor, feedback was unclear, and I was never informed.",
        "The workload was overwhelming with constant overtime and stress.",
        "The internship was unpaid and the stipend never covered my expenses.",
        "I was given menial tasks like getting coffee and no real project.",
    ]
    openers = ["", "Overall, ", "Honestly, ", "In my experience, ", "To be fair, "]
    closers = ["", " That sums it up.", " I would say this clearly.", " Just my opinion."]

    rows = []
    for label, pool in (("positive", pos), ("negative", neg)):
        for _ in range(n_per_class):
            text = random.choice(openers) + random.choice(pool) + random.choice(closers)
            if random.random() < 0.5:
                text += " " + random.choice(pool)
            rows.append({"feedback": text.strip(), "sentiment": label})
    return pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z\s']", " ", text)
    return " ".join(t for t in text.split() if t not in STOPWORDS and len(t) > 1)


def transformer_predict(texts):
    from transformers import pipeline
    clf = pipeline("sentiment-analysis",
                   model="distilbert-base-uncased-finetuned-sst-2-english",
                   truncation=True, max_length=256)
    return [r["label"].lower() for r in clf(texts, batch_size=32)]

In [16]:
if DATA_PATH is None:
    df = make_sample_data()
else:
    df = pd.read_csv(DATA_PATH)
    if "feedback" not in df.columns:
        raise ValueError("CSV must contain a 'feedback' column.")

df = df.dropna(subset=["feedback"]).drop_duplicates(subset=["feedback"]).copy()

if "sentiment" in df.columns:
    df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
elif "rating" in df.columns:
    df = df[df["rating"] != 3].copy()
    df["sentiment"] = df["rating"].ge(4).map({True: "positive", False: "negative"})
else:
    df["sentiment"] = transformer_predict(df["feedback"].tolist())

df = df[df["sentiment"].isin(["positive", "negative"])].copy()
df["clean"] = df["feedback"].apply(clean_text)
df = df[df["clean"].str.len() > 0].reset_index(drop=True)

print(f"Dataset size after cleaning: {len(df)}")
df["sentiment"].value_counts()

Dataset size after cleaning: 474


,count
sentiment,
negative,238
positive,236


In [17]:
df.head()

,feedback,sentiment,clean
0,"To be fair, The team culture was welcoming and...",positive,fair team culture welcoming felt included day ...
1,"Overall, The internship was unpaid and the sti...",negative,overall internship unpaid stipend never covere...
2,"Honestly, My mentor was unavailable and I rare...",negative,honestly mentor unavailable rarely received gu...
3,"Honestly, Great communication, regular feedbac...",positive,honestly great communication regular feedback ...
4,"Great communication, regular feedback, and I a...",positive,great communication regular feedback felt info...


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["sentiment"], test_size=0.2,
    stratify=df["sentiment"], random_state=SEED)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(vec.fit_transform(X_train), y_train)
preds = model.predict(vec.transform(X_test))

print(f"Accuracy: {accuracy_score(y_test, preds):.3f}\n")
print(classification_report(y_test, preds))

Accuracy: 1.000

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00        48
    positive       1.00      1.00      1.00        47

    accuracy                           1.00        95
   macro avg       1.00      1.00      1.00        95
weighted avg       1.00      1.00      1.00        95



In [19]:
cm = confusion_matrix(y_test, preds, labels=["negative", "positive"])
pd.DataFrame(cm,
             index=["actual negative", "actual positive"],
             columns=["predicted negative", "predicted positive"])

,predicted negative,predicted positive
actual negative,48,0
actual positive,0,47


In [20]:
weights = sorted(zip(vec.get_feature_names_out(), model.coef_[0]), key=lambda x: x[1])

pd.DataFrame({
    "top_negative_terms": [w for w, _ in weights[:15]],
    "top_positive_terms": [w for w, _ in weights[::-1][:15]],
})

,top_negative_terms,top_positive_terms
0,never,organized documentation
1,no,organized
2,did,onboarding organized
3,did not,helpful
4,not,documentation helpful
5,confusing,welcoming felt
6,confusing no,welcoming
7,no proper,included day
8,onboarding confusing,felt included
9,proper,day


In [21]:
neg = df[df["sentiment"] == "negative"]
rows = []
for theme, words in THEMES.items():
    pattern = r"\b(?:" + "|".join(map(re.escape, words)) + r")"
    count = int(neg["feedback"].str.lower().str.contains(pattern, regex=True).sum())
    rows.append({"theme": theme,
                 "negative_mentions": count,
                 "share_of_negative_%": round(100 * count / max(len(neg), 1), 1)})

improvement = pd.DataFrame(rows).sort_values("negative_mentions", ascending=False).reset_index(drop=True)
improvement.to_csv(os.path.join(OUT_DIR, "improvement_areas.csv"), index=False)
improvement

,theme,negative_mentions,share_of_negative_%
0,Learning & Meaningful Work,83,34.9
1,Compensation,53,22.3
2,Culture & Inclusion,48,20.2
3,Onboarding & Training,43,18.1
4,Workload & Work-Life Balance,43,18.1
5,Communication,39,16.4
6,Mentorship & Guidance,38,16.0
